# ViANLI Experiments — Colab workflow

Notebook này dùng riêng cho **ViANLI**, bài toán Natural Language Inference 3 lớp. Notebook tuân theo pipeline của `main_colab.ipynb`: lưu output vào Google Drive, clone repository, cài dependency, lấy secrets an toàn, rồi chạy training/evaluation bằng lệnh rõ ràng.

Validation macro F1 là metric phát triển/chọn checkpoint; chỉ đánh giá test sau khi đã khóa finalist.

## 1. Mount Google Drive

Checkpoint và kết quả được lưu vào Drive để không mất sau khi Colab runtime kết thúc. Bật GPU và Internet trước khi chạy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repository và cài dependencies

In [ ]:
PROJECT_DIR = '/content/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'

!rm -rf $PROJECT_DIR
!git clone --depth 1 --branch main $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
%pip install -q -r requirements.txt

## 3. Xác thực Hugging Face và Weights & Biases

Tạo Colab Secrets `HF_TOKEN` và `WANDB_API_KEY` trước khi chạy. Token không được ghi vào notebook. Output ViANLI được tách riêng khỏi ViHSD trong Drive.

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Create a Colab Secret named HF_TOKEN before continuing.')
os.environ['HF_TOKEN'] = hf_token

wandb_api_key = userdata.get('WANDB_API_KEY')
if not wandb_api_key:
    raise RuntimeError('Create a Colab Secret named WANDB_API_KEY before continuing.')
os.environ['WANDB_API_KEY'] = wandb_api_key

import wandb
wandb.login(verify=True)
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/ViANLI-MoE/checkpoints'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/ViANLI-MoE/results'
print('ViANLI Colab environment configured.')

## 4. Kiểm tra dữ liệu và preprocessing

ViANLI dùng `input_mode: nli`, tokenizer nhận riêng `premise` và `hypothesis`. XLM-R có thể đọc `uitnlp/ViANLI`. Nếu chạy PhoBERT, chuẩn bị bản segment trước:

```bash
python scripts/segment_vianli.py --output-dir data/vianli_segmented
```

Label order phải là `entailment=0`, `contradiction=1`, `neutral=2`; kiểm tra lại trong `resolved_config.yaml`.

## 5. Training commands

Chạy smoke test trước để kiểm tra dataset, tokenizer, model shape và output path. Smoke-test không phải kết quả thực nghiệm. Sau đó chạy từng experiment với `run_id` duy nhất.

In [ ]:
# Smoke test.
!python train.py --config configs/vianli.yaml --smoke-test --run-id vianli-smoke-s42

# Transformer-MoE candidate — last 6 layers, 4 experts, dynamic routing.
# !python train.py --config configs/finalists/vianli_best.yaml --no-smoke-test \
#   --run-id vianli-moe-last6-e4-dynamic-s42 --set seed=42

## 6. Repeated seeds và controlled comparisons

Khi candidate ổn định, lặp cùng cấu hình với seeds `42`, `43`, `44`. Chỉ thay một yếu tố mỗi comparison: backbone, architecture, MoE placement, expert count hoặc routing.

```bash
python train.py --config configs/finalists/vianli_best.yaml --no-smoke-test \
  --run-id vianli-moe-last6-e4-dynamic-s43 --set seed=43
```

Không kết luận từ một seed đơn; báo cáo mean ± standard deviation và efficiency/routing diagnostics nếu có.

## 7. Evaluation command

Chỉ chạy sau khi cấu hình/seed set đã được khóa. Evaluation lưu predictions, classification report, confusion matrix và class distributions vào `results/<run-id>/`.

In [ ]:
# Thay run_id bằng run đã train thành công.
# !python evaluate.py --final-evaluation --config configs/finalists/vianli_best.yaml --run-id vianli-moe-last6-e4-dynamic-s42

# Debug checkpoint cụ thể:
# !python evaluate.py --final-evaluation --config configs/finalists/vianli_best.yaml \
#   --checkpoint checkpoints/vianli-moe-last6-e4-dynamic-s42/transformer_moe_best.safetensors

## 8. Artifact, tracker và reference

Kiểm tra `checkpoints/<run-id>/resolved_config.yaml`, metadata, `results/<run-id>/run_metrics.json` và predictions. Cập nhật `experiments/experiment_tracker.csv` với status, verification, seed, platform (`Colab`), config path, validation/test metrics và artifact path. Test không dùng để chọn model; run bất thường giữ lại và đánh dấu `anomaly`.

| Thành phần | Giá trị |
|---|---|
| Dataset | `uitnlp/ViANLI` hoặc bản segment local |
| Input | `premise` + `hypothesis`, `input_mode=nli` |
| Labels | `entailment`, `contradiction`, `neutral` |
| Backbone chính | `xlm-roberta-base` |
| MoE candidate | layers `[6,7,8,9,10,11]` |
| Selection metric | validation macro F1 |
| Seeds xác nhận | 42, 43, 44 |